# 04 — Final Evaluation and Report

Final evaluation on the held-out test set. All decisions — the selected checkpoints, the decision thresholds and the preregistered comparisons — are fixed on validation before any test access. A visible Boolean guard and a plain protocol snapshot keep that ordering explicit.

## Final-evaluation guard

In [ ]:
from pathlib import Path
import json
import sys
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.final_evaluation import (
    REQUIRED_CHECKLIST, final_dataset_status, generate_publication_report,
    run_final_evaluation, save_protocol_snapshot,
)
from notebooks.utility.classifier_protocol import load_selected_generators
RUN_FINAL_EVALUATION = False
FINAL_DATASET_ADAPTER = None
final_dataset_status(FINAL_DATASET_ADAPTER)

## Readiness checklist

In [ ]:
CHECKLIST = {
    'Generator benchmark completed': False,
    'Generators selected': False,
    '24 downstream jobs completed': False,
    '8 validation ensembles completed': False,
    'Validation analysis finalized': False,
    'Checkpoints selected using validation only': False,
    'Decision thresholds selected using validation only': False,
    'Statistical comparisons declared': False,
    'Final evaluation dataset identified': False,
    'No further model selection will occur': False,
}
for item in REQUIRED_CHECKLIST:
    print('[x]' if CHECKLIST[item] else '[ ]', item)

## Preregistered primary comparisons

In [ ]:
PLANNED_COMPARISONS = [
    {'architecture': architecture, 'condition_a': left, 'condition_b': right, 'metric': 'pr_auc', 'multiplicity': 'Holm family of 8'}
    for architecture in ('maxvit512', 'mammofm')
    for left, right in (
        ('real_only', 'real_augmented'),
        ('real_only', 'real_plus_best_finetuned_positive'),
        ('real_only', 'real_plus_best_fromscratch_positive'),
        ('real_plus_best_finetuned_positive', 'real_plus_best_fromscratch_positive'),
    )
]
PLANNED_COMPARISONS

## Plain protocol snapshot

In [ ]:
SAVE_PROTOCOL_SNAPSHOT = False
FINAL_EVALUATION_DATASET_IDENTIFIER = ''
SEED_CHECKPOINTS = {}
VALIDATION_SELECTED_THRESHOLDS = {}
if SAVE_PROTOCOL_SNAPSHOT:
    selected = load_selected_generators(ROOT)
    snapshot_path = save_protocol_snapshot(
        ROOT, selected_generators=selected, seed_checkpoints=SEED_CHECKPOINTS,
        validation_thresholds=VALIDATION_SELECTED_THRESHOLDS,
        planned_comparisons=PLANNED_COMPARISONS,
        final_evaluation_dataset_identifier=FINAL_EVALUATION_DATASET_IDENTIFIER,
        notes='Decisions fixed before final evaluation; no later model selection.',
    )
    print(snapshot_path)
else:
    print('Protocol snapshot not saved.')

## Optional final evaluation

In [ ]:
# No final data is touched unless this flag is True and a real adapter is supplied.
if RUN_FINAL_EVALUATION:
    final_result = run_final_evaluation(ROOT, run_final_evaluation=True, checklist=CHECKLIST, adapter=FINAL_DATASET_ADAPTER)
else:
    final_result = None
    print('Final evaluation disabled: no final dataset was accessed.')

## Publication report

In [ ]:
GENERATE_REPORT = True
report_path = generate_publication_report(ROOT) if GENERATE_REPORT else ROOT / 'results/publication_v2/publication_report.md'
print(report_path if report_path.is_file() else 'Not yet evaluated')

## Interpretation and limitations

Primary inference uses patient-level bootstrap and Holm correction over the eight declared comparisons on the test set. Any additional analysis is exploratory. Checkpoints and thresholds are fixed on validation, so the test set contributes no model selection.